# Module 14 — Advanced Spark SQL

**Topics in this module:**
1. Plain JSON vs. Nested JSON
2. Syntax `:` for navigating a JSON string
3. Parsing JSON to StructType — `schema_of_json` + `from_json`
4. Interacting with Structs — `.` syntax
5. Flattening — `.*`, field by field, `explode()`

---
## Setup


In [0]:
# Environment variables
catalog = "main"
schema  = "school"
volume  = "raw_data"

base_path     = f"/Volumes/{catalog}/{schema}/{volume}"
students_path = f"{base_path}/students-json"

print("Base path:", base_path)
print("Students path:", students_path)


Base path: /Volumes/main/school/raw_data
Students path: /Volumes/main/school/raw_data/students-json


In [0]:
# Create structure if it does not exist
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")
print("List structure")

List structure


In [0]:
# Copy data from S3 to Volumes
s3_base = "s3://dalhussein-books/DEA-Book/datasets/school/v1"

dbutils.fs.cp(f"{s3_base}/students-json", students_path, recurse=True)
print("Data ready in:", students_path)

Data ready in: /Volumes/main/school/raw_data/students-json


In [0]:
# Create students table from JSON
spark.sql(f""" 
    CREATE TABLE IF NOT EXISTS {catalog}.{schema}.students 
    AS SELECT * 
    FROM json.`{students_path}`
""")
print("Table students list")

Table students list


---
## 1. Plain JSON vs. Nested JSON

The `profile` column contains a **JSON string** — Spark treats it as plain text.

Before we can operate on its fields, we need to parse it.

In [0]:
%sql
-- View the data: notice the profile column
SELECT * 
FROM main.school.students

email,gpa,profile,student_id,updated
dabby2y@japanpost.jp,1.48,"{""first_name"":""Dniren"",""last_name"":""Abby"",""gender"":""Female"",""address"":{""street"":""768 Mesta Terrace"",""city"":""Annecy"",""country"":""France""}}",S00001,2021-12-14T23:15:43.375Z
eabbysc1@github.com,3.02,"{""first_name"":""Etti"",""last_name"":""Abbys"",""gender"":""Female"",""address"":{""street"":""1748 Vidon Plaza"",""city"":""Varge Mondar"",""country"":""Portugal""}}",S00002,2021-12-14T23:15:43.375Z
rabelovd1@wikispaces.com,3.31,"{""first_name"":""Ronnie"",""last_name"":""Abelov"",""gender"":""Male"",""address"":{""street"":""363 Randy Park"",""city"":""San Celestio"",""country"":""Philippines""}}",S00003,2021-12-14T23:15:43.375Z
rabels9g@behance.net,1.89,"{""first_name"":""Ray"",""last_name"":""Abels"",""gender"":""Female"",""address"":{""street"":""613 Lyons Way"",""city"":""Oudtshoorn"",""country"":""South Africa""}}",S00004,2021-12-14T23:15:43.375Z
sabendrothin@cargocollective.com,3.55,"{""first_name"":""Shanon"",""last_name"":""Abendroth"",""gender"":""Female"",""address"":{""street"":""30292 Manufacturers Junction"",""city"":""Ani-e"",""country"":""Philippines""}}",S00005,2021-12-14T23:15:43.375Z
null,2.9,"{""first_name"":""Norman"",""last_name"":""Abernethy"",""gender"":""Male"",""address"":{""street"":""9292 Oxford Center"",""city"":""Gibara"",""country"":""Cuba""}}",S00006,2021-12-14T23:15:43.375Z
sabrahmson3h@blinklist.com,2.96,"{""first_name"":""Skell"",""last_name"":""Abrahmson"",""gender"":""Male"",""address"":{""street"":""90941 Hallows Park"",""city"":""Huarong Chengguanzhen"",""country"":""United Kingdom""}}",S00007,2021-12-14T23:15:43.375Z
dacheson2h@mapy.cz,1.2,"{""first_name"":""Darsey"",""last_name"":""Acheson"",""gender"":""Non-binary"",""address"":{""street"":""29579 Grim Plaza"",""city"":""Dārayyā"",""country"":""Syria""}}",S00008,2021-12-14T23:15:43.375Z
fackwoodji@gravatar.com,1.96,"{""first_name"":""Fredrick"",""last_name"":""Ackwood"",""gender"":""Male"",""address"":{""street"":""67 Dunning Plaza"",""city"":""Santo Domingo"",""country"":""Cuba""}}",S00009,2021-12-14T23:15:43.375Z
null,1.39,"{""first_name"":""Doralynne"",""last_name"":""Adamkiewicz"",""gender"":""Female"",""address"":{""street"":""84126 Glendale Center"",""city"":""Ugep"",""country"":""Nigeria""}}",S00010,2021-12-14T23:15:43.375Z


In [0]:
%sql
-- Confirm that profile is a STRING, not a structured type
DESCRIBE main.school.students

col_name,data_type,comment
email,string,null
gpa,double,null
profile,string,null
student_id,string,null
updated,string,null


---
## 2. Syntax `:` — navigating a JSON string

> **Rule:** Use `:` when the column is a **JSON string**.
> Use it to quickly extract 1-2 fields. For complete pipelines → `from_json`.

| Syntax | When to use it |
|---|---|
`column:field` | The column is a **JSON string** |
`column.field` | The column is a native **StructType** |

In [0]:
%sql
-- First level access with two points
SELECT 
  student_id, 
  profile:first_name, 
  profile:address:country
FROM main.school.students

student_id,first_name,country
S00001,Dniren,France
S00002,Etti,Portugal
S00003,Ronnie,Philippines
S00004,Ray,South Africa
S00005,Shanon,Philippines
S00006,Norman,Cuba
S00007,Skell,United Kingdom
S00008,Darsey,Syria
S00009,Fredrick,Cuba
S00010,Doralynne,Nigeria


---
## 3. Parsing JSON to StructType

A **StructType** is Spark's native, binary representation:
- Direct access without re-parsing text in each operation
- Well-defined data types
- Optimized by the Catalyst engine

**2-step flow:**
1. `schema_of_json()` — infers the schema from an example
2. `from_json()` — applies the schema and converts the column to StructType


In [0]:
%sql
-- STEP 1 + 2: create view with the parsed struct
CREATE OR REPLACE TEMP VIEW parsed_students AS 
  SELECT 
    student_id, 
    from_json( 
    profile, 
    schema_of_json('{"first_name":"Sarah","last_name":"Lundi", 
    "gender":"Female","address":{"street":"8 Greenbank Rd", 
    "city":"Ottawa","country":"Canada"}}') 
    ) AS profile_struct 
  FROM main.school.students

In [0]:
%sql
-- Confirm that profile_struct is now a STRUCT (not a STRING)
DESCRIBE parsed_students

col_name,data_type,comment
student_id,string,null
profile_struct,"struct,first_name:string,gender:string,last_name:string>",null


In [0]:
%sql
-- View the data; the column is now expandable in the UI
SELECT * 
FROM parsed_students

student_id,profile_struct
S00001,"List(List(Annecy, France, 768 Mesta Terrace), Dniren, Female, Abby)"
S00002,"List(List(Varge Mondar, Portugal, 1748 Vidon Plaza), Etti, Female, Abbys)"
S00003,"List(List(San Celestio, Philippines, 363 Randy Park), Ronnie, Male, Abelov)"
S00004,"List(List(Oudtshoorn, South Africa, 613 Lyons Way), Ray, Female, Abels)"
S00005,"List(List(Ani-e, Philippines, 30292 Manufacturers Junction), Shanon, Female, Abendroth)"
S00006,"List(List(Gibara, Cuba, 9292 Oxford Center), Norman, Male, Abernethy)"
S00007,"List(List(Huarong Chengguanzhen, United Kingdom, 90941 Hallows Park), Skell, Male, Abrahmson)"
S00008,"List(List(Dārayyā, Syria, 29579 Grim Plaza), Darsey, Non-binary, Acheson)"
S00009,"List(List(Santo Domingo, Cuba, 67 Dunning Plaza), Fredrick, Male, Ackwood)"
S00010,"List(List(Ugep, Nigeria, 84126 Glendale Center), Doralynne, Female, Adamkiewicz)"


---
## 4. Interacting with Structs; `.` Syntax

> **Golden Rule:** with a StructType, always use a **dot (.)** to navigate.
> ⚠️ **Most Common Mistake:** mixing `:` (JSON string) with `.` (Struct). They are different things.

In [0]:
%sql
-- Access with point, first and second level
SELECT
  student_id,
  profile_struct.first_name,
  profile_struct.address.country
FROM parsed_students


student_id,first_name,country
S00001,Dniren,France
S00002,Etti,Portugal
S00003,Ronnie,Philippines
S00004,Ray,South Africa
S00005,Shanon,Philippines
S00006,Norman,Cuba
S00007,Skell,United Kingdom
S00008,Darsey,Syria
S00009,Fredrick,Cuba
S00010,Doralynne,Nigeria


In [0]:
%sql
-- Filter by nested field
SELECT *
FROM parsed_students
WHERE profile_struct.address.country = 'Canada'


student_id,profile_struct
S00017,"List(List(Bridgewater, Canada, 978 Roxbury Junction), Aprilette, Female, Agron)"
S00023,"List(List(Pemberton, Canada, 92145 Blue Bill Park Alley), Maryrose, Female, Algar)"
S00052,"List(List(Asbestos, Canada, 871 Monument Parkway), Gal, Male, Aspling)"
S00066,"List(List(Saint John, Canada, 25 Green Crossing), Analise, Female, Babbe)"
S00089,"List(List(Barrie, Canada, 8846 Dahle Court), Ezmeralda, Female, Bartrap)"
S00109,"List(List(Chester, Canada, 43 3rd Trail), Dan, Male, Bedward)"
S00216,"List(List(L'Île-Perrot, Canada, 542 Merchant Alley), Eberhard, Male, Bunton)"
S00268,"List(List(Sherwood Park, Canada, 71847 Graedel Trail), Alverta, Female, Chastand)"
S00308,"List(List(Osoyoos, Canada, 7910 Delladonna Street), Inesita, Female, Collough)"
S00347,"List(List(Lachute, Canada, 1377 Michigan Lane), Henderson, Male, Crook)"


In [0]:
%sql
-- Group by nested field
SELECT
  profile_struct.address.country,
  COUNT(*) AS total_estudiantes
FROM parsed_students
GROUP BY profile_struct.address.country
ORDER BY total_estudiantes DESC


country,total_estudiantes
United Kingdom,319
Indonesia,162
Philippines,89
Russia,79
Brazil,68
Poland,64
Portugal,52
France,52
Japan,38
Sweden,36


---
## 5. Flattening; Flattening the Structure

Converts the fields of the Struct into separate **top-level columns**.
Necessary for exporting to CSV, connecting BI tools, or simplifying pipelines.

| Method | Speed ​​| Control | When to Use It |
---|---|---|---|
`struct.*` | High | Low | Exploration, Prototyping |
| Field by Field | Medium | Total | Production, with aliases |
`explode()` | Medium | High | When there are arrays |

### 5a. Wildcard `.*` — más rápido (un nivel a la vez)


In [0]:
%sql
-- Level 1: expands profile_struct (address remains struct)
CREATE OR REPLACE TEMP VIEW students_flat AS
SELECT student_id, profile_struct.*
FROM parsed_students;

In [0]:
%sql
SELECT * FROM students_flat;

student_id,address,first_name,gender,last_name
S00001,"List(Annecy, France, 768 Mesta Terrace)",Dniren,Female,Abby
S00002,"List(Varge Mondar, Portugal, 1748 Vidon Plaza)",Etti,Female,Abbys
S00003,"List(San Celestio, Philippines, 363 Randy Park)",Ronnie,Male,Abelov
S00004,"List(Oudtshoorn, South Africa, 613 Lyons Way)",Ray,Female,Abels
S00005,"List(Ani-e, Philippines, 30292 Manufacturers Junction)",Shanon,Female,Abendroth
S00006,"List(Gibara, Cuba, 9292 Oxford Center)",Norman,Male,Abernethy
S00007,"List(Huarong Chengguanzhen, United Kingdom, 90941 Hallows Park)",Skell,Male,Abrahmson
S00008,"List(Dārayyā, Syria, 29579 Grim Plaza)",Darsey,Non-binary,Acheson
S00009,"List(Santo Domingo, Cuba, 67 Dunning Plaza)",Fredrick,Male,Ackwood
S00010,"List(Ugep, Nigeria, 84126 Glendale Center)",Doralynne,Female,Adamkiewicz


In [0]:
%sql
-- Level 2: also expand address
SELECT student_id, first_name, last_name, gender, address.*
FROM students_flat;


student_id,first_name,last_name,gender,city,country,street
S00001,Dniren,Abby,Female,Annecy,France,768 Mesta Terrace
S00002,Etti,Abbys,Female,Varge Mondar,Portugal,1748 Vidon Plaza
S00003,Ronnie,Abelov,Male,San Celestio,Philippines,363 Randy Park
S00004,Ray,Abels,Female,Oudtshoorn,South Africa,613 Lyons Way
S00005,Shanon,Abendroth,Female,Ani-e,Philippines,30292 Manufacturers Junction
S00006,Norman,Abernethy,Male,Gibara,Cuba,9292 Oxford Center
S00007,Skell,Abrahmson,Male,Huarong Chengguanzhen,United Kingdom,90941 Hallows Park
S00008,Darsey,Acheson,Non-binary,Dārayyā,Syria,29579 Grim Plaza
S00009,Fredrick,Ackwood,Male,Santo Domingo,Cuba,67 Dunning Plaza
S00010,Doralynne,Adamkiewicz,Female,Ugep,Nigeria,84126 Glendale Center


### 5b. Field by field with aliases; recommended in production


In [0]:
%sql
SELECT
  student_id,
  profile_struct.first_name       AS first_name,
  profile_struct.last_name        AS last_name,
  profile_struct.address.city     AS city,
  profile_struct.address.country  AS country
FROM parsed_students


student_id,first_name,last_name,city,country
S00001,Dniren,Abby,Annecy,France
S00002,Etti,Abbys,Varge Mondar,Portugal
S00003,Ronnie,Abelov,San Celestio,Philippines
S00004,Ray,Abels,Oudtshoorn,South Africa
S00005,Shanon,Abendroth,Ani-e,Philippines
S00006,Norman,Abernethy,Gibara,Cuba
S00007,Skell,Abrahmson,Huarong Chengguanzhen,United Kingdom
S00008,Darsey,Acheson,Dārayyā,Syria
S00009,Fredrick,Ackwood,Santo Domingo,Cuba
S00010,Doralynne,Adamkiewicz,Ugep,Nigeria


### 5c. `explode()`; when the Struct contains an array

> ⚠️ `explode()` **multiplies the rows** — one row per array element.
> Keep this in mind before performing any JOINs or subsequent aggregations.

"courses": ["Math", "Science", "History"]

In [0]:
%sql
-- We create a view with JSON that includes an array of courses
CREATE OR REPLACE TEMP VIEW students_with_courses AS
SELECT * FROM VALUES
  (1, '{"first_name":"Ana",  "address":{"country":"Canada"}, "courses":["Math","Science","History"]}'),
  (2, '{"first_name":"Luis", "address":{"country":"Mexico"}, "courses":["Math","Art"]}'),
  (3, '{"first_name":"Sara", "address":{"country":"Spain"},  "courses":["Science"]}')
AS t(student_id, profile)


In [0]:
%sql
CREATE OR REPLACE TEMP VIEW students_courses_parsed AS
SELECT
  student_id,
  from_json(
    profile,
    schema_of_json('{"first_name":"Ana","address":{"country":"Canada"},"courses":["Math"]}')
  ) AS profile_struct
FROM students_with_courses

In [0]:
%sql
SELECT
  student_id,
  profile_struct.first_name,
  explode(profile_struct.courses) AS course
FROM students_courses_parsed

student_id,first_name,course
1,Ana,Math
1,Ana,Science
1,Ana,History
2,Luis,Math
2,Luis,Art
3,Sara,Science


---
## Clean Up


In [0]:
def clean_up(): 
    print("Deleting tables...") 
    spark.sql("DROP TABLE IF EXISTS main.school.students") 

    print("Deleting volume...") 
    dbutils.fs.rm(base_path, True) 

    print("Deleting schema...") 
    spark.sql(f"DROP SCHEMA IF EXISTS {catalog}.{schema} CASCADE") 

    print("Done")

In [0]:
clean_up()

Deleting tables...
Deleting volume...
Deleting schema...
Done
